# Manager notebook -- yolo11 waste-detection fine-tune

Thin Colab orchestrator over the `trashmonkey` package: zero logic in
cells, only imports and calls (manager-notebook architecture). Designed to
survive "Runtime > Run all" on a fresh Colab session. Fill in the credential
constants in the Init cell before running on Colab.

## 1. Init

Constants, Colab detection, Drive mount, and a preflight that collects every
problem before a single assert. `torch`/`ultralytics` are required on Colab
only (they ship with the runtime); locally their absence is a warning.

In [ ]:
# Init: config constants, Colab detection, Drive mount, preflight validation.
import importlib
import os
import sys
from pathlib import Path

GITHUB_USERNAME = ""  # fill in: GitHub account with read access to the repo
GITHUB_TOKEN = ""  # fill in: personal access token (repo read scope)
REPO_URL = ""  # fill in: e.g. https://github.com/<user>/TrashMonkey.git

# Kaggle API token (kaggle.com -> Settings -> API -> Create New Token). ONLY
# needed the first time the dataset is built; once it is cached on Drive every
# later run restores the archive and never touches Kaggle. Leave blank if you
# already uploaded ~/.kaggle/kaggle.json to the runtime.
KAGGLE_USERNAME = ""
KAGGLE_KEY = ""

IN_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")
DRIVE_BASE = Path("/content/drive/MyDrive/yolo-waste-sorter")  # historical name kept: existing Drive checkpoints/runs resume from here
MIN_ULTRALYTICS = (8, 3, 226)  # train(augmentations=[...]) floor; training guards enforce it too

if IN_COLAB and not os.path.ismount("/content/drive"):
    from google.colab import drive

    drive.mount("/content/drive")
if IN_COLAB:
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)

# Export Kaggle creds to the env so the kaggle CLI in the build stage finds them
# (only consulted on a cache miss; a warm Drive cache needs no Kaggle access).
if KAGGLE_USERNAME and KAGGLE_KEY:
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
    os.environ["KAGGLE_KEY"] = KAGGLE_KEY

errors = []
warnings = []
required = ["yaml", "PIL", "numpy", "albumentations"]
colab_only = ["torch", "ultralytics"]  # ship with Colab; optional on a local machine
for module_name in required + colab_only:
    try:
        importlib.import_module(module_name)
    except ImportError as exc:
        if module_name in colab_only and not IN_COLAB:
            warnings.append(f"{module_name} not importable locally; the train/eval cells need it")
        else:
            errors.append(f"package not importable: {module_name} ({exc})")

if IN_COLAB:
    credentials = {
        "GITHUB_USERNAME": GITHUB_USERNAME,
        "GITHUB_TOKEN": GITHUB_TOKEN,
        "REPO_URL": REPO_URL,
    }
    for name, value in credentials.items():
        if not value:
            errors.append(f"{name} is empty -- fill in the constants at the top of this cell")

try:
    import ultralytics

    parsed = tuple(int(part) for part in ultralytics.__version__.split(".")[:3])
    if parsed < MIN_ULTRALYTICS:
        floor = ".".join(str(part) for part in MIN_ULTRALYTICS)
        errors.append(
            f"ultralytics {ultralytics.__version__} < {floor}: "
            "train(augmentations=[...]) needs the validated hook added in 8.3.226"
        )
except ImportError:
    pass  # absence already handled above (error on Colab, warning locally)

for warning in warnings:
    print(f"WARNING: {warning}")
assert not errors, "preflight failed:\n- " + "\n- ".join(errors)
print(f"preflight ok: IN_COLAB={IN_COLAB}, DRIVE_BASE={DRIVE_BASE if IN_COLAB else 'unused locally'}")


## 2. Clone / update repo

Idempotent: clones on the first run, fast-forward pulls afterwards. Local runs
locate the repo root instead of cloning.

In [ ]:
# Clone (Colab) or locate (local) the repo, then put src/ on sys.path.
import os
import subprocess
import sys
from pathlib import Path

if IN_COLAB:
    REPO_ROOT = Path("/content/TrashMonkey")
    # Build the authenticated URL in parts (the credential-in-URL literal
    # pattern would trip the repo's secret-scanning push gate).
    auth = GITHUB_USERNAME + ":" + GITHUB_TOKEN
    auth_url = REPO_URL.replace("https://", "https://" + auth + "@", 1)
    if (REPO_ROOT / ".git").is_dir():
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", auth_url, str(REPO_ROOT)], check=True)
else:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    REPO_ROOT = next(
        path
        for path in candidates
        if (path / "pyproject.toml").is_file() and (path / "src" / "trashmonkey").is_dir()
    )

os.chdir(REPO_ROOT)
SRC_DIR = str(REPO_ROOT / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
print(f"repo root: {REPO_ROOT}")


## 3. Dataset: build-or-restore (Drive-cached)

Self-contained: on a **cache miss** the notebook runs the full data pipeline
in-process (download -> remap -> autobox -> qa -> dedup -> balance -> split) and
saves the result to Drive; on a **cache hit** it just restores the archive --
no rebuild. The cache key is a fingerprint of `datasets.yaml` plus the
dataset-relevant config fields (classes, seed, eval split), so a cap or
registry edit rebuilds automatically while an unchanged dataset is never
re-combined. Building needs Kaggle credentials (Init cell) the first time only.

In [ ]:
# Build the processed dataset on a cache miss, else restore it from Drive.
# restore_or_build fingerprints the inputs and picks: local (warm) -> restored
# (Drive archive matches) -> rebuilt (build + archive to Drive). save=IN_COLAB
# pushes the archive to Drive only on Colab; locally it just builds in place.
import os
from pathlib import Path

from trashmonkey.data.cache import restore_or_build
from trashmonkey.data.pipeline import build_dataset
from trashmonkey.utils.config import load_config

cfg = load_config()
CONFIG_PATH = REPO_ROOT / "configs" / "config.yaml"
DATASETS_PATH = REPO_ROOT / "configs" / "datasets.yaml"
DATA_ROOT = REPO_ROOT / "data"
DATASET_YAML = REPO_ROOT / cfg.paths.processed / cfg.experiment.name / "dataset.yaml"
SPLIT_MANIFEST = REPO_ROOT / cfg.paths.interim / "pipeline" / "split.yaml"

# Persistent store: Google Drive on Colab, a local cache dir otherwise.
CACHE_BASE = (DRIVE_BASE / "data") if IN_COLAB else (DATA_ROOT / "cache")
ARCHIVE = CACHE_BASE / "processed.tar.gz"
FINGERPRINT = CACHE_BASE / "processed.fingerprint"
LOCAL_MARKER = DATA_ROOT / ".dataset.fingerprint"


def _build():
    # Kaggle creds are consulted ONLY here, on an actual (re)build.
    has_token = (Path.home() / ".kaggle" / "kaggle.json").is_file()
    has_env = bool(os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
    if not (has_token or has_env):
        raise RuntimeError(
            "dataset build needs Kaggle credentials: set KAGGLE_USERNAME/KAGGLE_KEY "
            "in the Init cell, or upload a token to ~/.kaggle/kaggle.json. "
            "(Not needed once the built dataset is cached on Drive.)"
        )
    # ack_review=True: unattended continue past the QA gate (documented mechanism).
    build_dataset(CONFIG_PATH, ack_review=True)


decision = restore_or_build(
    config_path=CONFIG_PATH,
    datasets_path=DATASETS_PATH,
    data_root=DATA_ROOT,
    dataset_yaml=DATASET_YAML,
    archive_path=ARCHIVE,
    fingerprint_path=FINGERPRINT,
    local_marker=LOCAL_MARKER,
    build_fn=_build,
    save=IN_COLAB,
)
print(f"dataset {decision.status} (fingerprint {decision.fingerprint[:12]})")
print(f"dataset yaml:   {DATASET_YAML}")
print(f"archive:        {ARCHIVE if IN_COLAB else '(local build -- not archived)'}")
assert DATASET_YAML.is_file(), f"dataset yaml missing after {decision.status}: {DATASET_YAML}"
print(
    f"split manifest: {SPLIT_MANIFEST} "
    f"(exists={SPLIT_MANIFEST.is_file()}; the evaluate cell requires it)"
)


## 4. Smoke test

One tiny CPU epoch through the full train cycle. `train(smoke=True)`
synthesizes its own dataset when `data_yaml` is None and pins
device=cpu/imgsz=160/batch=2 internally.

In [ ]:
# Smoke test: full train cycle on CPU against a synthesized tiny dataset.
from trashmonkey.models.training import train
from trashmonkey.utils.config import load_config

cfg = load_config()
smoke_result = train(cfg, data_yaml=None, smoke=True)
assert smoke_result.best_pt.is_file(), f"smoke run produced no best.pt: {smoke_result.best_pt}"
print(f"smoke ok: best_pt={smoke_result.best_pt}")
print(f"smoke mAP50={smoke_result.metrics['map50']:.4f} run_dir={smoke_result.run_dir}")


## 5. Checkpoint / run detection

Lists prior run directories on the persistence root. The package's `train()`
exposes no resume parameter, so a fresh run is started either way (reported
as an API gap by task 013).

In [ ]:
# Prior-run detection on the persistence root: pick up an interrupted run.
# find_resumable() reads the epoch field inside each last.pt -- ultralytics
# strips finished runs to epoch=-1, so only genuinely interrupted runs match.
from trashmonkey.models.training import find_resumable

RUNS_ROOT = (DRIVE_BASE / "runs") if IN_COLAB else (REPO_ROOT / "experiments" / "runs")
RESUME_FROM = find_resumable(RUNS_ROOT)
if RESUME_FROM is not None:
    print(f"interrupted run detected -- training will resume from {RESUME_FROM}")
else:
    print(f"no interrupted run under {RUNS_ROOT}; training starts fresh")


## 5b. Runtime profile (auto-batch)

Detects whatever accelerator this session landed on and resolves `batch` /
`workers` from a static VRAM table (`training/autobatch.py`). Deterministic --
the same card always yields the same recipe, and the resolved integers are
logged in `runs.jsonl` -- so it is reproducible, unlike ultralytics
`batch=-1`. `batch` is capped at 64 (ultralytics' nominal batch size), so a
bigger GPU only changes throughput, not the optimisation. On a local CPU
session it falls back to the pinned `config.yaml` default (batch 16, workers 8).

In [ ]:
# Auto-batch: detect the accelerator once and resolve batch/workers for it.
# RUNTIME is reused by the train cell so detection happens exactly once.
from trashmonkey.models.training import detect_runtime

RUNTIME = detect_runtime()
print(f"runtime: {RUNTIME.summary()}")
if RUNTIME.device == "cpu":
    print("WARNING: no CUDA device -- a full 100-epoch run on CPU is impractical; "
          "this path is for the local smoke test only.")


## 6. Train

Config-driven T7 fine-tune. On Colab the run directory and `runs.jsonl` live
on Drive so checkpoints and the experiment log survive disconnects.

In [ ]:
# T7 fine-tune; checkpoints and the run log persist to Drive on Colab, so a
# disconnect loses nothing and the next "Run all" resumes via RESUME_FROM.
# apply_runtime() stamps the auto-detected batch/workers onto the config; the
# resolved values are logged in runs.jsonl like any other pinned arg.
from trashmonkey.models.training import DEFAULT_RUNS_JSONL, apply_runtime, train
from trashmonkey.utils.config import load_config

cfg, _ = apply_runtime(load_config(), RUNTIME)
print(f"training with batch={cfg.train.batch}, workers={cfg.train.workers} ({RUNTIME.summary()})")
RUNS_JSONL = (DRIVE_BASE / "experiments" / "runs.jsonl") if IN_COLAB else DEFAULT_RUNS_JSONL
run = train(
    cfg, data_yaml=DATASET_YAML, runs_jsonl=RUNS_JSONL, project=RUNS_ROOT, resume=RESUME_FROM
)
print(f"best_pt:    {run.best_pt}")
print(f"run_dir:    {run.run_dir}")
print(f"mAP50:      {run.metrics['map50']:.4f}")
for name, block in run.metrics["per_class"].items():
    print(f"  {name:<12} map50={block['map50']:.4f} recall={block['recall']:.4f}")
print(f"runs.jsonl: {RUNS_JSONL}")


## 7. Evaluate

Three-tier evaluation (VAL -> TEST-1 -> TEST-2 severity curve) plus the T7
escalation verdict that drives the yolo11n -> yolo11s decision.

In [ ]:
# T6 three-tier evaluation + T7 escalation verdict (nano -> small decision).
from trashmonkey.models.evaluation import evaluate
from trashmonkey.utils.config import load_config

cfg = load_config()
assert SPLIT_MANIFEST.is_file(), (
    f"split manifest missing: {SPLIT_MANIFEST}\n"
    "make repro writes it to data/interim/pipeline/split.yaml; include "
    "interim/pipeline in the uploaded archive (see the dataset cell)."
)
report = evaluate(cfg, run.best_pt, DATASET_YAML, SPLIT_MANIFEST)

header = f"{'class':<12}{'precision':>10}{'recall':>9}{'map50':>9}{'map50-95':>10}{'conf@p95':>10}"
for tier in (report.val, report.test1, *report.test2):
    print(
        f"\n[{tier.tier}] split={tier.split} severity={tier.severity} "
        f"map50={tier.map50:.4f} map50-95={tier.map50_95:.4f}"
    )
    print(header)
    for name, cls_eval in sorted(tier.per_class.items()):
        conf = "never" if cls_eval.conf_at_p95 is None else f"{cls_eval.conf_at_p95:.3f}"
        print(
            f"{name:<12}{cls_eval.precision:>10.4f}{cls_eval.recall:>9.4f}"
            f"{cls_eval.map50:>9.4f}{cls_eval.map50_95:>10.4f}{conf:>10}"
        )

escalation = report.escalation
print("\n" + "=" * 76)
if escalation["passed"]:
    print("ESCALATION VERDICT: PASS -- yolo11n meets the T7 floors; stay on nano")
else:
    print("ESCALATION VERDICT: FAIL -- T7 floors missed; consider yolo11n -> yolo11s")
print(
    f"overall mAP50 {escalation['overall_map50']:.4f} "
    f"(floor {escalation['thresholds']['overall_map50']})"
)
for name, block in escalation["per_class"].items():
    map50 = "absent" if block["map50"] is None else f"{block['map50']:.4f}"
    recall = "absent" if block["recall"] is None else f"{block['recall']:.4f}"
    print(f"  {name:<12} map50={map50} recall={recall} passed={block['passed']}")
print("=" * 76)


## 8. Plots

Supplied by task 014 (`trashmonkey/visualization/plots.py`); skipped
cleanly until it lands.

In [ ]:
# Plot stage (task 014): render every figure the run artifacts support.
from pathlib import Path

from trashmonkey.utils.config import load_config
from trashmonkey.visualization import plots

cfg = load_config()
PLOTS_DIR = (DRIVE_BASE / "plots") if IN_COLAB else (REPO_ROOT / "reports" / "figures")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


def _save(name, render):
    out = PLOTS_DIR / name
    render(out)
    print(f"saved {out}")


_save("dataset_composition.png", lambda o: plots.plot_dataset_composition(SPLIT_MANIFEST, save_path=o))
_save("tier_comparison.png", lambda o: plots.plot_tier_comparison(report, save_path=o))
_save("severity_curve.png", lambda o: plots.plot_severity_curve(report, save_path=o))
_save(
    "per_class_curves_val.png",
    lambda o: plots.plot_per_class_curves(report, Path(report.val.curves_path), save_path=o),
)

results_csv = run.run_dir / "results.csv"
if results_csv.is_file():
    _save("training_curves.png", lambda o: plots.plot_training_curves(results_csv, save_path=o))
else:
    print(f"skip training_curves: {results_csv} missing")

eval_dir = Path(report.detections_path).parent
wilderness = eval_dir / "wilderness_detections.jsonl"
if wilderness.is_file():
    _save(
        "openset_separation.png",
        lambda o: plots.plot_confidence_separation(
            Path(report.detections_path), wilderness, save_path=o,
            tau_frame=cfg.thresholds.tau_frame,
        ),
    )
else:
    print(f"skip openset_separation: {wilderness} missing (thresholds stage dumps it)")

sweep_csv = eval_dir.parent / "thresholds" / "sweep.csv"
if sweep_csv.is_file():
    _save("threshold_tradeoff.png", lambda o: plots.plot_threshold_tradeoff(sweep_csv, save_path=o))
else:
    print(f"skip threshold_tradeoff: {sweep_csv} missing (run models.thresholds first)")

sample = next(iter(sorted((DATASET_YAML.parent / "images" / "val").glob("*.jpg"))), None)
if sample is not None:
    _save("degradation_grid.png", lambda o: plots.plot_degradation_grid(sample, save_path=o, seed=cfg.seed))
else:
    print("skip degradation_grid: no val image found beside DATASET_YAML")

## 9. Summary

In [ ]:
# Run summary.
from pathlib import Path

from trashmonkey.models.evaluation import REPORT_FILENAME
from trashmonkey.utils.config import load_config

cfg = load_config()
print(
    f"experiment:  {cfg.experiment.name} (seed={cfg.seed}, base={cfg.model.base}, "
    f"imgsz={cfg.model.imgsz}, epochs={cfg.train.epochs}, batch={RUNTIME.batch}, "
    f"workers={RUNTIME.workers}, optimizer={cfg.train.optimizer}, lr0={cfg.train.lr0})"
)
print(f"runtime:     {RUNTIME.summary()}")
print(f"best mAP50:  {run.metrics['map50']:.4f}")
for name, block in run.metrics["per_class"].items():
    print(f"  {name:<12} map50={block['map50']:.4f} recall={block['recall']:.4f}")
verdict = "PASS (stay on yolo11n)" if report.escalation["passed"] else "FAIL (consider yolo11s)"
print(f"escalation:  {verdict}")
print(f"run dir:     {run.run_dir}")
print(f"eval report: {Path(report.detections_path).parent / REPORT_FILENAME}")
print(f"plots dir:   {PLOTS_DIR}")
print(f"runs.jsonl:  {RUNS_JSONL}")
